# Chapter 55: Multimodal Application Development
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo  
**Objective:** Merancang dan mengimplementasikan pipeline aplikasi pemantauan cerdas berpusat visi (*vision-centric intelligent monitoring*) yang terintegrasi secara end-to-end, mengevaluasi fusi sadar-keandalan multi-sensor, menguji ketahanan fail-safe saat sensor dropout, serta memprofil latensi per frame dan throughput FPS untuk deployment pada perangkat keras tepi (*Edge AI*).

### 1. Landasan Rekayasa Pipeline Sistem & Formulasi Latensi
Model Latensi End-to-End Per-Frame:
$$T_{\text{total}} = T_{\text{capture}} + \max(T_{\text{enc, } v}, T_{\text{enc, } a}, T_{\text{enc, } t}) + T_{\text{sync}} + T_{\text{fuse}} + T_{\text{post}}$$
$$\text{Throughput (FPS)} = \frac{1000}{T_{\text{total}} \text{ (ms)}}$$

Fusi Keputusan Adaptif Sadar-Keandalan:
$$\mathbf{P}_{\text{fused}} = \sum_{m \in \{v, a, t\}} \alpha_m \mathbf{P}_m, \quad \alpha_m = \frac{\gamma_m}{\sum_k \gamma_k}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy murni.")

In [ ]:
# 3. Implementasi Pipeline Pemantauan & Validasi Seksi 3 (Skenario A, B, C)
def run_multimodal_monitoring(p_matrix, gamma_list):
    gammas = np.array(gamma_list, dtype=np.float64)
    probs = np.array(p_matrix, dtype=np.float64)
    tot = np.sum(gammas)
    weights = gammas / tot if tot > 0 else np.ones_like(gammas) / len(gammas)
    p_fused = np.sum(weights[:, np.newaxis] * probs, axis=0)
    return p_fused, weights, int(np.argmax(p_fused))

# Data Uji: 3 Kelas [0: Normal, 1: Penyusupan, 2: Bahaya]
P_v = [0.15, 0.75, 0.10]
P_a = [0.20, 0.70, 0.10]
P_t = [0.10, 0.80, 0.10]

# Skenario A: Operasi Normal
pf_a, w_a, pred_a = run_multimodal_monitoring([P_v, P_a, P_t], [0.85, 0.60, 0.75])
print(f"Skenario A (Normal): P_fused(1) = {pf_a[1]:.4f} | Prediksi: Kelas {pred_a}")
np.testing.assert_allclose(pf_a[1], 0.753409, atol=1e-5)
assert pred_a == 1

# Skenario B: Dropout Kamera Tertutup Asap (gamma_v = 0.0)
pf_b, w_b, pred_b = run_multimodal_monitoring([P_v, P_a, P_t], [0.0, 0.60, 0.75])
print(f"Skenario B (Dropout): P_fallback(1) = {pf_b[1]:.4f} | Prediksi: Kelas {pred_b}")
np.testing.assert_allclose(pf_b[1], 0.755556, atol=1e-5)
assert pred_b == 1

# Skenario C: Profiling Latensi Frame
t_cap = 2.5
t_enc = max([14.0, 4.0, 2.0])
t_sync, t_fuse, t_post = 0.5, 0.8, 1.2
t_tot = t_cap + t_enc + t_sync + t_fuse + t_post
fps = 1000.0 / t_tot
print(f"Skenario C (Latensi): {t_tot:.2f} ms | Throughput: {fps:.2f} FPS")
assert t_tot == 19.0
np.testing.assert_allclose(fps, 52.631579, atol=1e-4)
print("VALIDASI PIPELINE & LATENSI: Seluruh skenario analitik Seksi 3 lulus sempurna!")

In [ ]:
# 4. Simulasi Streaming Asinkron & Antrean Bounded Ring Buffer
from collections import deque

class AsyncSensorQueue:
    def __init__(self, maxlen=10):
        self.buffer = deque(maxlen=maxlen)
    def push_packet(self, timestamp, payload):
        self.buffer.append({'ts': timestamp, 'data': payload})
    def pop_synchronized(self, target_ts, tolerance_ms=50.0):
        best_pkt = None
        best_diff = float('inf')
        for pkt in self.buffer:
            diff = abs(pkt['ts'] - target_ts)
            if diff < best_diff and diff <= tolerance_ms:
                best_diff = diff
                best_pkt = pkt
        return best_pkt

queue = AsyncSensorQueue(maxlen=5)
queue.push_packet(100.0, np.array([0.1, 0.2]))
queue.push_packet(133.3, np.array([0.3, 0.4]))
queue.push_packet(166.7, np.array([0.5, 0.6]))

synced_pkt = queue.pop_synchronized(135.0, tolerance_ms=10.0)
print("Paket Tersinkronisasi Terdekat (Target=135.0 ms):", synced_pkt)
assert synced_pkt is not None and synced_pkt['ts'] == 133.3
print("VERIFIKASI ASINKRON BUFFER: Lulus deterministik.")

In [ ]:
# 5. Pola Deployment Edge AI (ONNX Runtime / TensorRT Spec)
if HAS_TORCH:
    print("Mode PyTorch / GPU Aktif.")
    dummy_input = torch.randn(1, 3, 224, 224)
    print("Dummy Input Tensor Visual:", dummy_input.shape)
else:
    print("PANDUAN PENYEBARAN SISTEM MULTIMODAL DI LINGKUNGAN PRODUKSI:")
    print("1. Konversi Model ke Format Terstandarisasi ONNX:")
    print("   torch.onnx.export(model, (dummy_v, dummy_a), 'multimodal_system.onnx', ...)")
    print("2. Kuantisasi INT8 Pasca-Pelatihan (TensorRT / ONNXRuntime):")
    print("   from onnxruntime.quantization import quantize_dynamic, QuantType")
    print("   quantize_dynamic('model.onnx', 'model_int8.onnx', weight_type=QuantType.QInt8)")
    print("3. Arsitektur Kontainerisasi:")
    print("   - Thread 1: RTSP Video Capture (GStreamer / FFmpeg)")
    print("   - Thread 2: ALSA Audio Capture (PyAudio / sounddevice)")
    print("   - Thread 3: ZeroMQ / gRPC Message Dispatcher ke Dashboard Web")

### 6. Rangkuman & Pembahasan Akademik
1. **Sintesis Holistik**: Menghubungkan teori ekstraksi fitur, penyelarasan temporal, dan fusi sadar-keandalan ke dalam arsitektur aplikasi industri yang kokoh.
2. **Jaminan Real-Time**: Profiling latensi end-to-end membuktikan bahwa sistem mampu beroperasi pada $>50$ FPS di perangkat keras tepi.
3. **Ketahanan Fail-Safe**: Desain sensor fallback menjamin operasional sistem pemantauan tanpa henti bahkan saat terjadi sensor dropout ekstrem.